## Imports

In [ ]:
import numpy as np
import xarray as xr
import copy
import src.utils
import src.lme_utils
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cmocean
import os
import pathlib
import cartopy.crs as ccrs
import pandas as pd
import scipy.stats
import tqdm
import time
import xesmf as xe

## Set seaborn plotting style
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})
sns.set_palette("colorblind")

## initialize RNG
rng = np.random.default_rng(seed=100)

## Funcs

In [ ]:
## specify lons/lats for E/W boxes
LONS_W = [120, 200]
LONS_E = [200, 280]


## funcs to plot boxes
def plot_wbox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_W, lats=[-5, 5], **kwargs)


def plot_ebox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_E, lats=[-5, 5], **kwargs)


## funcs to average over regions
def get_eq_avg(data, lons):
    """average over equatorial region and longitudes"""
    return data.sel(longitude=slice(*lons), latitude=slice(-5, 5)).mean(
        ["longitude", "latitude"]
    )


def get_e(data):
    return get_eq_avg(data, LONS_E)


def get_w(data):
    return get_eq_avg(data, LONS_W)


def get_dx(data):
    return get_e(data) - get_w(data)


def add_gridlines(axs):
    """func to add gridlines to axs object"""

    for ax in axs[:-1, 0]:
        ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[],
            ylocs=[-30, 0, 30],
        )
    for ax in axs[-1]:
        gl_ = ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[50, 120 - 160],
            # xlocs=[],
            ylocs=[-30, 0, 30],
        )
        gl_.top_labels = False
        # gl_.bottom_labels = True
    # gl_.left_labels = False

    return


def plot_level(ax, data, level, ls="-", c="white"):
    """plot single level on hovmoller"""
    ax.contour(
        data.longitude,
        data.latitude,
        data,
        levels=[level],
        colors=c,
        linestyles=ls,
        zorder=10,
        transform=ccrs.PlateCarree(),
    )
    return


import cartopy.crs as ccrs


def make_cb_range(dx, nlevs):
    amp = dx * nlevs - dx / 2

    lev1 = np.arange(0, amp + dx / 2, dx) + dx / 2
    lev0 = -lev1[::-1]
    return np.concatenate([lev0, lev1])


def plot_pr_diff(ax, data, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.balance_r",
        levels=make_cb_range(dx, nlev),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp


def plot_sst_sigma(ax, data, lev_min=0.3, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.amp",
        levels=np.arange(lev_min, lev_min + dx * (nlev + 1), dx),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp


def plot_pr(ax, data, lev_min=1, dx=1, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.rain",
        levels=np.arange(lev_min, lev_min + dx * (nlev + 1), dx),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp


def prep(y):
    """prepare data for EOF analysis"""

    ## resample to seasonal
    y_ = y.resample({"time": "QS-JAN"}).mean()

    ## find months
    is_jas = y_.time.dt.month == 7
    is_ond = y_.time.dt.month == 10
    is_jfm = y_.time.dt.month == 1
    in_range = is_jas.values | is_ond.values | is_jfm.values

    ## subset for data
    y_ = y_.isel(time=in_range).isel(time=slice(1, -2))

    ## get year start
    yrs = y_.time.dt.year.values
    mths = y_.time.dt.month.values
    year_start = np.array([y - 1 if (m == 1) else y for (y, m) in zip(yrs, mths)])

    ## get season
    season_dict = {1: "JFM", 7: "JAS", 10: "OND"}
    season = np.array([season_dict[m] for m in mths])

    ## create multi-index
    new_idx = pd.MultiIndex.from_arrays([year_start, season], names=["y0", "season"])

    ## convert to xr
    new_idx_xr = xr.Coordinates.from_pandas_multiindex(new_idx, dim="time")

    ## add to original data
    y_ = y_.assign_coords(new_idx_xr)

    return y_.unstack("time")


def fill_between_months(ax, m0, m1):
    """shade month range"""

    ## get bounds for plot
    N = 10
    yrange = ax.get_ylim()

    ## get bounds for plot
    y = np.linspace(*yrange, N)
    x1 = (m0 - 0.2) * np.ones(N)
    x2 = (m1 + 0.2) * np.ones(N)

    ## plot
    ax.fill_betweenx(y=y, x1=x1, x2=x2, color="gray", edgecolor="white", alpha=0.2)

    return

## Data loading

In [ ]:
def sel_area(data, lon_range=None, lat_range=None):
    """select area on t grids"""

    ## find in lon range
    if lon_range is None:
        lon_idx = np.array([True for _ in data.nlon])
    else:
        lon_idx = (
            ((data.TLONG >= lon_range[0]) & (data.TLONG <= lon_range[1]))
            .any("nlat")
            .values
        )

    if lat_range is None:
        lat_idx = np.array([True for _ in data.nlat])
    else:
        lat_idx = (
            ((data.TLAT >= lat_range[0]) & (data.TLAT <= lat_range[1]))
            .any("nlon")
            .values
        )

    return data.isel(nlon=lon_idx, nlat=lat_idx)


def area_avg(data, varname, lon_range=None, lat_range=None):
    """average over area"""

    ## trim data
    data_ = sel_area(data, lon_range=lon_range, lat_range=lat_range)

    ## get dims to sum over
    integrate = lambda x: (x * data_["dA"]).sum(["nlon", "nlat"])

    return integrate(data_[varname]) / integrate(1.0)


def get_eli_helper(exceeds_thresh):
    """compute ELI from mask of threshold exceedance"""

    ## sum and count longitudes exceeding thresh
    lon = exceeds_thresh.longitude
    longitude_sum = (exceeds_thresh * lon).sum(["longitude", "latitude"])
    longitude_count = exceeds_thresh.sum(["longitude", "latitude"])

    ## eli is average longitude
    eli = longitude_sum / longitude_count

    return eli


def get_eli(rsst, rsst_thresh=0, max_lat=15):
    """compute ELI from tropical SST data"""

    ## get equatorial Pac. SST
    rsst_pac = rsst.sel(longitude=slice(120, 280), latitude=slice(-max_lat, max_lat))

    ## get boolean array where SST exceeds thresh
    exceeds_thresh0 = rsst_pac >= rsst_thresh

    ## compute initial ELI estimate
    eli0 = get_eli_helper(exceeds_thresh0)

    return eli0

## Load the data

### init. cluster

In [ ]:
from dask.distributed import LocalCluster, Client

cluster = LocalCluster(n_workers=2)
client = Client(cluster)
client

### load SST data

In [ ]:
SAVE_FP_SNR = pathlib.Path("/glade/work/kcarr/lme_data/snr")

## load
time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

## load sst
sst = xr.open_dataset(SAVE_FP_SNR / "sst_split.nc", decode_times=time_coder)
sst = sst.rename({"lat": "latitude", "lon": "longitude"}).compute()

## load rsst
rsst = xr.open_dataset(SAVE_FP_SNR / "rsst_split.nc", decode_times=time_coder)

## load precip
pr = xr.open_dataset(SAVE_FP_SNR / "pr_split.nc", decode_times=time_coder)
pr = pr.rename({"lat": "latitude", "lon": "longitude"}).compute()

## make sure time coords match
pr = pr.assign_coords({"time": sst.time})

### Compute indices

In [ ]:
def get_pr_idx(x, is_n=False):
    """compute pr indices"""

    ## longitude range
    lon_range = slice(65, 140)

    ## average over latss
    if is_n:
        lat_range = slice(5, 25)
    else:
        lat_range = slice(-25, -5)

    ## select data
    x_ = x.sel(latitude=lat_range, longitude=lon_range)

    return x_.mean(["latitude", "longitude"])

#### compute

In [ ]:
idx = xr.merge(
    [
        src.utils.get_nino3(sst["anom"]).rename("T_3"),
        src.utils.get_nino34(sst["anom"]).rename("T_34"),
        src.utils.get_nino4(sst["anom"]).rename("T_4"),
        src.utils.get_nino3(rsst["total"]).rename("rT_3"),
        src.utils.get_nino34(rsst["total"]).rename("rT_34"),
        src.utils.get_nino4(rsst["total"]).rename("rT_4"),
        get_pr_idx(pr["anom"], is_n=True).rename("pr_n"),
        get_pr_idx(pr["anom"], is_n=False).rename("pr_s"),
        get_eli(rsst["total"]).rename("eli"),
    ]
)

## trim to pre-industrial period
idx = prep(idx).sel(y0=slice(900, 1850))

## Analysis

Seasonal cycle of precip

In [ ]:
get_clim = lambda x: x.groupby("time.month").mean()

fig, ax = plt.subplots(figsize=(4, 3))

## plot data
for b, l in zip([True, False], ["n", "s"]):
    ax.plot(np.arange(1, 13), get_clim(get_pr_idx(pr["forced"], is_n=b)), label=l)

## should seasons
fill_between_months(ax, 1, 3)
fill_between_months(ax, 7, 9)
ax.legend()

plt.show()

Seasonal correlation

In [ ]:
## plot
fig, axs = plt.subplots(1, 2, figsize=(8, 4), layout="constrained")
for ax, la in zip(axs, ["pr_n", "pr_s"]):
    ax.scatter(
        idx[la].sel(season="JAS"),
        idx[la].sel(season="JFM"),
        s=3,
    )
    zz = np.linspace(-2, 2)
    ax.plot(zz, -zz, c="k", lw=1)
    ax.set_xlabel("ASO rain")
    ax.set_title(la)

## formatting
axs[0].set_ylabel("FMA rain")
src.utils.set_lims(axs)
axs[1].set_yticks([])

plt.show()

Relation to ENSO

In [ ]:
## Get Niño index
x0 = idx["T_4"].sel(season="OND")

for re in ["pr_n", "pr_s"]:

    ## specify plot region
    y = idx[re]

    ## plot
    fig, axs = plt.subplots(1, 4, figsize=(12, 3), layout="constrained")

    ## kwargs for scatter
    sc_kwargs = dict(s=3, alpha=0.5)

    ## plot difference
    for ax, sea in zip(axs[:-1], ["JAS", "JFM"]):
        x1 = y.sel(season=sea)
        corr = xr.corr(x0, x1).values.item()
        ax.scatter(x0, x1, **sc_kwargs)
        ax.set_title(f"{sea}, $r=${corr:.2f}")

    ## plot sum
    x1 = y.sel(season="JAS") + y.sel(season="JFM")
    corr = xr.corr(x0, x1).values.item()
    axs[-2].scatter(x0, 0.5 * x1, **sc_kwargs)
    axs[-2].set_title(f"JAS$+$JFM, $r=${corr:.2f}")

    ## plot difference
    x1 = y.sel(season="JFM") - y.sel(season="JAS")
    corr = xr.corr(x0, x1).values.item()
    axs[-1].scatter(x0, 0.5 * x1, **sc_kwargs)
    axs[-1].set_title(f"JFM$-$JAS, $r=${corr:.2f}")

    ## formatting
    src.utils.set_lims(axs)
    for ax in axs[1:]:
        ax.set_yticks([])
    for ax in axs:
        ax.set_xlabel(r"$T_4$")
        ax.axhline(0, ls="--", c="k", lw=0.8)
        ax.axvline(0, ls="--", c="k", lw=0.8)

    axs[0].set_ylabel("mm/day")
    axs[-1].text(s=re, x=1.1, y=0.45, transform=axs[-1].transAxes)

    plt.show()

PDFs of Niño 4 and ELI

In [ ]:
edges0 = np.linspace(-2.8, 2.5, 20)
pdf0, _ = src.utils.get_empirical_pdf(
    idx["T_4"].sel(season="OND").values.flatten(), edges=edges0,
)

edges1 = np.linspace(178, 205, 20)
pdf1, _ = src.utils.get_empirical_pdf(
    idx["eli"].sel(season="OND").values.flatten(), edges=edges1,
)


fig,axs = plt.subplots(1,2,figsize=(8,3), layout="constrained")

## Plot T4 PDF
axs[0].stairs(pdf0, edges0)
axs[0].set_xlabel(r"$T_4$")
src.utils.add_vticks(axs[:1], xticks=[-2,0,2],xlines=[0])

## Plot ELI PDF
axs[1].stairs(pdf1, edges1)
axs[1].set_xlabel(r"ELI")
# src.utils.add_vticks(axs[:1], xticks=[-2,0,2],xlines=[0])

plt.show()

Scatter: ELI vs. $T_4$  

Bimodal distribution b/c for extreme El Niño $T_4$ starts decreasing

In [ ]:
x0 = idx["eli"].sel(season="OND")
x1 = idx["T_4"].sel(season="OND")

fig,ax = plt.subplots(figsize=(4,4))
ax.scatter(x0, x1, **sc_kwargs)
# src.utils.add_vticks([ax], xticks=[-2,0,2],xlines=[0])
axs

plt.show()